In [1]:
import sys
import os
import pandas as pd
from dotenv import load_dotenv  
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
#project_root = Path.cwd().resolve().parent
#sys.path.insert(0, str(project_root))
#print("Added to sys.path:", project_root)

from constants import NAN_VALUE, AnnotationsReduced, AnnotatedReportReduced
from train_utils import create_list_of_annotated_reports
from model_utils import (
    get_binary_classification_fields,
    get_classification_fields,
    get_multiple_choice_fields,
    get_regression_fields
)

import json
from mistralai import Mistral
import time
from IPython.display import clear_output
from ast import literal_eval


In [9]:
# Set the API key for Mistral
load_dotenv()  # Load environment variables from .env file
mistral_api_key = os.getenv("MISTRAL_API_KEY")

# Initialize the Mistral client
client = Mistral(api_key=mistral_api_key)

In [10]:
# Parameters
TRAIN_FILE_NAME = "train_mistral.jsonl"
VALIDATION_FILE_NAME = "validation_mistral.jsonl"
TEST_FILE_NAME = "test_mistral.jsonl"

TRAINED_MODEL_ID = 'ft:classifier:ministral-3b-latest:6f88df17:20260106:037d39cd'

In [6]:
reg_fields = get_regression_fields(AnnotationsReduced)
cl_fields = get_classification_fields(AnnotationsReduced)
mc_fields = get_multiple_choice_fields(AnnotationsReduced)
bc_fields = get_binary_classification_fields(AnnotationsReduced)
print("Campi di regressione:", reg_fields)
print("Campi di classificazione:", cl_fields)
print("Campi di multi-scelta:", mc_fields)
print("Campi binari:", bc_fields)    

Campi di regressione: []
Campi di classificazione: ['morfologia', 'riflessione_peritoneale_anteriore', 'infiltrazione_tessuto_adiposo', 'stadio_T']
Campi di multi-scelta: ['posizione', 'infiltrazione_organi_dettagli', 'sedi_linfonodi']
Campi binari: ['infiltrazione_sfinteri', 'infiltrazione_organi_extra', 'coinvolgimento_riflessione_peritoneale', 'coinvolgimento_fascia_mesorettale', 'depositi_tumorali', 'emvi_esteso', 'stadio_N', 'stadio_N1c', 'mrf', 'emvi', 'metastasi']


In [ ]:
# Carichiamo i nostri file csv
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
    'test': TEST_FILE_NAME,
}

paths = {
    split: Path('../MISTRAL/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    # Load the samples
    with open(path, "r") as f:
        data[split] = [json.loads(l) for l in f.readlines()]

train_data, validation_data, test_data = data['train'], data['validation'], data['test']

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")

len(train_data) = 166
len(validation_data) = 54
len(test_data) = 55


In [8]:
test_data[0]

{'text': "RM ADDOME INFERIORE (S/C MDC)\nINDICAZIONE ALL'ESAME: RM DI STADIAZIONE IN PAZIENTE CON RECENTE RISCONTRO ALLA COLONSCOPIA DI LESIONE ETEROFORMATIVA DEL RETTO - ESAME ISTOLOGICO IN CORSO.\nTECNICA: ESAME ESEGUITO CON SEQUENZE FSE, FRFSE, DWI E FSPGR 3D-DISCO, SECONDO PIANI MULTIPLI, PRIMA E DOPO SOMMINISTRAZIONE EV DI MDC PARAMAGNETICO (PROHANCE 0.2 ML/KG), PREVIA DISTENSIONE RETROGRADA DEL RETTO CON GEL.\nRISULTATI: PRESENZA DI GROSSOLANO ISPESSIMENTO PARIETALE CIRCONFERENZIALE, IPERINTENSO NELLE SEQUENZE T2 DIPENDENTI E CON SEGNI DI NETTA RESTRIZIONE DELLA DIFFUSIONE PROTONICA, CHE INTERESSA A TUTTO SPESSORE LA PARETE DEL RETTO MEDIO-ALTO PER UN TRATTO DI ALMENO 8 CM (SPESSORE DA 8 MM A 21 MM), IL CUI MARGINE INFERIORE DISTA CIRCA 3.6 CM DA OAI. LA LESIONE MOSTRA SEGNI DI SCONFINAMENTO NEL TESSUTO ADIPOSO PERIVISCERALE, CON GETTONI DI TESSUTO SOLIDO LUNGO IL PROFILO POSTERO-LATERALE DESTRO (H 6-8) E LUNGO I VASI EMORROIDARI SUPERIORI, CON SOTTILI TRALCI CHE RAGGIUNGONO IL P

In [11]:
# Classify the first test sample
classifier_response = client.classifiers.classify(
    model=TRAINED_MODEL_ID,
    inputs=[test_data[0]["text"]],
)
print("Text:", test_data[0]["text"])
print("Classifier Response:", json.dumps(classifier_response.model_dump(), indent=4))

Text: RM ADDOME INFERIORE (S/C MDC)
INDICAZIONE ALL'ESAME: RM DI STADIAZIONE IN PAZIENTE CON RECENTE RISCONTRO ALLA COLONSCOPIA DI LESIONE ETEROFORMATIVA DEL RETTO - ESAME ISTOLOGICO IN CORSO.
TECNICA: ESAME ESEGUITO CON SEQUENZE FSE, FRFSE, DWI E FSPGR 3D-DISCO, SECONDO PIANI MULTIPLI, PRIMA E DOPO SOMMINISTRAZIONE EV DI MDC PARAMAGNETICO (PROHANCE 0.2 ML/KG), PREVIA DISTENSIONE RETROGRADA DEL RETTO CON GEL.
RISULTATI: PRESENZA DI GROSSOLANO ISPESSIMENTO PARIETALE CIRCONFERENZIALE, IPERINTENSO NELLE SEQUENZE T2 DIPENDENTI E CON SEGNI DI NETTA RESTRIZIONE DELLA DIFFUSIONE PROTONICA, CHE INTERESSA A TUTTO SPESSORE LA PARETE DEL RETTO MEDIO-ALTO PER UN TRATTO DI ALMENO 8 CM (SPESSORE DA 8 MM A 21 MM), IL CUI MARGINE INFERIORE DISTA CIRCA 3.6 CM DA OAI. LA LESIONE MOSTRA SEGNI DI SCONFINAMENTO NEL TESSUTO ADIPOSO PERIVISCERALE, CON GETTONI DI TESSUTO SOLIDO LUNGO IL PROFILO POSTERO-LATERALE DESTRO (H 6-8) E LUNGO I VASI EMORROIDARI SUPERIORI, CON SOTTILI TRALCI CHE RAGGIUNGONO IL PIANO FA

In [12]:
458 + 44 + 273 + 223

998

In [13]:
67+667+29+46+189

998

In [17]:
classifier_response.results[0]['mrf'].scores

{'True': 0.4647800922393799, 'False': 0.5352199077606201}

In [31]:
classifier_response.model_dump()['results'][0]['mrf']['scores']

{'True': 0.4647800922393799, 'False': 0.5352199077606201}